## Automated sales outreach SDR team

## Part 1 :- Email setup

In [29]:
## env eke smtp server name daganna oni,app password ekath dagnna oni,apema email ekath danna oni, openai api key ekath daganna oni

In [30]:
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, ModelSettings
import os
import asyncio
import smtplib
from email.message import EmailMessage

In [31]:
load_dotenv(override=True)

True

In [32]:
import os
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

PROVIDER = "gemini"  # "gemini", "groq", හෝ "openai" තෝරන්න

if PROVIDER == "gemini":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.getenv("GEMINI_API_KEY"),
    )
    MODEL = OpenAIChatCompletionsModel(
        model="gemini-3.5-flash-lite",
        openai_client=external_client
    )

elif PROVIDER == "groq":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
    )
    MODEL = OpenAIChatCompletionsModel(
        model="openai/gpt-oss-120b",
        openai_client=external_client
    )

else:
    # OpenAI පාවිච්චි කරනවා නම්
    MODEL = "gpt-4.1-mini"

In [33]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

In [34]:
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [35]:
send_email("Test Email", "This is a test email sent from the Python script.", "<p>This is a test email sent from the Python script.</p>")

## Part 2 :- Ocastrate Via code

In [36]:
instructions1 = """
You are a sales agent working for Ocean Breeze Hotel,
a hotel offering comfortable stays and memorable guest experiences.
You write emails introducing new services, including spa treatments,
airport transfers, rooftop dining, and guided local tours.
Your email style is professional, elegant, and welcoming,
emphasizing comfort and service quality.
"""

instructions2 = """
You are a sales agent working for Ocean Breeze Hotel,
a hotel offering comfortable stays and memorable guest experiences.
You write emails introducing new services, including spa treatments,
airport transfers, rooftop dining, and guided local tours.
Your email style is witty, engaging, and humorous,
making guests excited to try the new services.
"""

instructions3 = """
You are a sales agent working for Ocean Breeze Hotel,
a hotel offering comfortable stays and memorable guest experiences.
You write emails introducing new services, including spa treatments,
airport transfers, rooftop dining, and guided local tours.
Your email style is concise and direct,
highlighting the new services and how to book them.
"""

In [37]:
sales_agent1 = Agent(name="Proffesional Sales Agent", instructions=instructions1, model=MODEL)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL)

In [38]:
results = Runner.run_streamed(sales_agent1, "Write a sales cold email.")

async for event in results.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Elevate Your Stay at Ocean Breeze Hotel 🌴

Dear [Guest Name],

We hope this email finds you well. 

At Ocean Breeze Hotel, we believe that true hospitality is about more than just a place to sleep—it is about creating moments that linger long after you check out. Whether you are visiting for a much-needed getaway or a productive business retreat, our goal is to ensure every moment of your stay is marked by comfort, elegance, and effortless relaxation.

To make your upcoming travels even more seamless and indulgent, we are delighted to introduce a curated collection of new guest experiences designed to elevate your time with us:

*   **Seamless Airport Transfers:** Begin your journey the moment you land. Let our professional chauffeurs welcome you at arrivals and transport you directly to our doors in comfort and style.
*   **Rejuvenating Spa Treatments:** Unwind and restore your senses with our newly expanded menu of holistic massages and wellness therapies, tailored to melt a

In [39]:
message = "Write a sales cold email."

with trace("parallel cold email generation"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

In [40]:
outputs = []

for result in results:
    outputs.append(result.final_output)

In [41]:
for output in outputs:
    print("\n\n---\n\n")
    print(output)



---


Subject: Elevate Your Upcoming Stay at Ocean Breeze Hotel 🌊

Dear [Prospect Name],

Whether your travels bring you to our shores for business or a well-deserved escape, we believe that every journey should be accompanied by moments of pure relaxation and effortless luxury. 

Here at **Ocean Breeze Hotel**, we take pride in offering not just a place to stay, but a truly memorable guest experience tailored to your every need. To ensure your upcoming visit is nothing short of exceptional, we are delighted to introduce our newly enhanced suite of guest services:

*   **Rejuvenating Spa Treatments:** Unwind and restore your senses with our bespoke massages and wellness therapies, designed to melt away the stresses of travel.
*   **Seamless Airport Transfers:** Begin and end your journey in comfort and style with our private, dependable chauffeur service.
*   **Exquisite Rooftop Dining:** Savor gourmet culinary creations paired with panoramic ocean views at our signature rooftop rest

In [42]:
## Methanin hodama email eka dan pick karaganna agent kenek set karganna oni

decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

In [43]:
sales_picker = Agent(name="Sales Email Picker", instructions=decision, model=MODEL)

In [44]:
messages = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, messages),
        Runner.run(sales_agent2, messages),
        Runner.run(sales_agent3, messages),
    )

    outputs = []

    for result in results:
        outputs.append(result.final_output)

    emails = "Coles sales emails:\n\n" + "\n\n".join(outputs)

    best_email_result = await Runner.run(sales_picker, emails)

    print("Best cold sales email:\n\n" + best_email_result.final_output)

Best cold sales email:

Subject: Step away from the spreadsheet, Gary. (An urgent invitation from Ocean Breeze Hotel) 🏖️

Dear [Guest Name],

Let’s play a quick game of association. 

When you hear the phrase "business trip," what comes to mind? 
A) A thrilling game of "How many lukewarm coffees can I consume before my heart gives out?"
B) Staring longingly at hotel wallpaper while wearing your business socks in bed.
C) Wondering if room service counts as a balanced diet (spoiler: it doesn't, but we respect the hustle).

If you answered C, or honestly *any* of the above, we need to talk. 

You’re booked in for a stay at the Ocean Breeze Hotel soon, and while our beds are so comfortable they border on legally binding, we refuse to let your stay be *just* another place to sleep. Life’s too short for boring hotel stays. 

That’s why we’ve rolled out a few new upgrades designed to rescue you from the ordinary:

*   **The "I Need to Untie My Shoulders" Spa:** Our masseuses have hands blesse

In [45]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Sends an email with the given subject, text body, and HTML body.

    Args:
        subject (str): The subject of the email.
        text_body (str): The plain text body of the email.
        html_body (str): The HTML body of the email.
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully."

In [46]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

In [47]:
required_tools = ModelSettings(tool_choice="required")

sales_sender = Agent(name="sales_agent", instructions=decision, model=MODEL, model_settings=required_tools, tools=[send_email_tool])

In [48]:
messages = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, messages),
        Runner.run(sales_agent2, messages),
        Runner.run(sales_agent3, messages),
    )

    outputs = []

    for result in results:
        outputs.append(result.final_output)

    emails = "Coles sales emails:\n\n" + "\n\n".join(outputs)

    best_email_result = await Runner.run(sales_sender, emails)

    print("Best cold sales email:\n\n" + best_email_result.final_output)

Best cold sales email:

As a customer, I would definitely respond best to the second email ("Don’t make us send the seagulls"). Most cold sales emails are dry, formal, and sound completely robotic. The humor, playful threat about seagulls, and relatable touches like the "Undo Adulting" spa make it stand out instantly in a crowded inbox and bring a genuine smile to your face. 

I have successfully selected and sent that email for you!


## Part 3 :- Ocastrate via LLM  mekata wadi barak denna




In [49]:
description = """
Use this tool to write a promotional email introducing Ocean Breeze Hotel's new services.
In the input, specify the services to promote.
"""


In [50]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

In [51]:
tools

[FunctionTool(name='sales_email_writer_1', description="\nUse this tool to write a promotional email introducing Ocean Breeze Hotel's new services.\nIn the input, specify the services to promote.\n", params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000161688F75C0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='sales_email_writer_2', description="\nUse this tool to write a promotional email introducing Ocean Breeze Hotel's new services.\nIn

In [52]:
## instruction tkay, task tkay wena wenama pahadiliwa ap liyaganna oni mekedi


instructions = """
You are a Sales Manager at Ocean Breeze Hotel.
Your goal is to select the single best promotional email introducing
the hotel's new services using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools
to create a promotional email introducing the hotel's new spa treatments,
airport transfers, rooftop dining, and guided local tours.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email
based on clarity, appeal to hotel guests, and a clear invitation to book.

3. Send the Email: Use your email-sending tool to send only the selected
email to the user. Send exactly one email.
"""


In [53]:
sales_manager = Agent(name="Sales Manager", instructions=instructions, model=MODEL, tools=tools, model_settings=ModelSettings(parallel_tool_calls=False))

In [54]:
with trace("Sales Manager Task"):
    response = await Runner.run(sales_manager, task)

    print("Sales Manager Result:\n\n" + response.final_output)

Sales Manager Result:

I have successfully evaluated the drafts and sent the selected email introducing Ocean Breeze Hotel's new services (spa treatments, airport transfers, rooftop dining, and guided local tours).
